# Production Method Comparison (Economic Analysis)

Live economic comparison of vanilla and modded production methods, grouped by slots.

In [61]:
import pandas as pd
import numpy as np

from core.parser.path_resolver import PathResolver
from core.data.building_data import BuildingData, PM_COMPARE_META_KEYS
from core.data.goods_data import GoodsData
from core.data.defines_data import DefinesData
from core.data.pop_data import PopData
from analysis.building_levels.building_analysis import load_config

config = load_config()
path_resolver = PathResolver(config['game_path'], config['mod_path'])
goods_data = GoodsData(path_resolver)
building_data = BuildingData(path_resolver)
defines_data = DefinesData(path_resolver)
pop_data = PopData(path_resolver)

goods_data.load_all()
building_data.load_all()
defines_data.load_all()
pop_data.load_all()

pd.options.display.max_columns = None
pd.options.display.precision = 2
pd.options.display.float_format = '{:.2f}'.format

In [62]:
# Resolved food good (good with food > 0) and its price; fallback to defines
food_good_vanilla = goods_data.get_food_good(False)
food_good_modded = goods_data.get_food_good(True)
food_price_vanilla = goods_data.get_food_good_price(False)
food_price_modded = goods_data.get_food_good_price(True)
if food_price_vanilla is None:
    food_price_vanilla = defines_data.get_vanilla_define("NMarket", "FOOD_PRICE")
if food_price_modded is None:
    food_price_modded = defines_data.get_define("NMarket", "FOOD_PRICE")
print(f"Vanilla food good: {food_good_vanilla}, price: {food_price_vanilla}")
print(f"Modded food good:  {food_good_modded}, price: {food_price_modded}")

Vanilla food good: rice, price: 1.0
Modded food good:  victuals, price: 3.0


In [63]:
pop_comparison = pd.merge(
    pop_data.vanilla_df[['pop_food_consumption']].rename(columns={'pop_food_consumption': 'Vanilla_Food'}),
    pop_data.modded_df[['pop_food_consumption']].rename(columns={'pop_food_consumption': 'Modded_Food'}),
    left_index=True,
    right_index=True
)
pop_comparison['Change'] = (pop_comparison['Modded_Food'] / pop_comparison['Vanilla_Food']) - 1
pop_comparison.style.format({'Change': '{:+.1%}'}).background_gradient(subset=['Change'], cmap='RdYlGn_r')

,Vanilla_Food,Modded_Food,Change
name,,,
nobles,20.000000,25.000000,+25.0%
clergy,5.000000,7.000000,+40.0%
burghers,4.000000,7.500000,+87.5%
laborers,1.000000,1.700000,+70.0%
soldiers,5.000000,5.000000,+0.0%
peasants,1.000000,1.000000,+0.0%
tribesmen,0.000000,0.000000,+nan%
slaves,1.000000,0.500000,-50.0%


In [64]:
# goods_data.vanilla_df.columns
# print(goods_data.vanilla_df.shape)
goods_data.vanilla_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).head(15)
# goods_data.vanilla_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).head(40)


# for element in goods_data.vanilla_df.reset_index().name.tolist():
#     print(element)

,method,default_market_price,transport_cost,food
name,,,,
rice,farming,1.00,1.00,10.00
maize,farming,1.00,1.00,8.00
wheat,farming,1.00,1.00,8.00
livestock,farming,1.50,1.00,8.00
potato,farming,1.00,1.00,8.00
fish,gathering,1.00,1.00,5.00
legumes,farming,1.00,2.00,5.00
wool,farming,1.25,1.00,5.00
millet,farming,1.00,1.00,5.00


In [65]:
# goods_data.modded_df.columns
# print(goods_data.modded_df.shape)
# print(goods_data.modded_df.columns)
goods_data.modded_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).tail(15)
# goods_data.modded_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="transport_cost", ascending=True).head(40)

,method,default_market_price,transport_cost,food
name,,,,
wool,farming,1.25,1.00,0.00
fur,hunting,2.00,1.00,0.00
wild_game,hunting,1.00,1.00,0.00
wheat,farming,1.00,1.00,0.00
maize,farming,1.00,1.00,0.00
rice,farming,1.00,1.00,0.00
fish,gathering,1.00,1.00,0.00
millet,farming,1.00,1.00,0.00
legumes,farming,1.00,2.00,0.00


In [66]:
def get_slot_df(building_name, slot_index, is_modded=True, comp=None):
    """If `comp` is passed, avoids re-running compare for each slot (same as analyze_building)."""
    if comp is None:
        comp = building_data.compare_production_methods(
            building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data
        )
    slots = comp['modded_slots'] if is_modded else comp['vanilla_slots']

    if slot_index >= len(slots):
        return pd.DataFrame()

    b_def = building_data.get_building(building_name) if is_modded else building_data.get_vanilla_building(building_name)
    b_def = b_def or {}

    slot = slots[slot_index]
    rows = []

    for pm_name, pm in slot.items():
        row = {
            "Version": "Modded" if is_modded else "Vanilla",
            "PM": pm_name,
            "Pop_Type": b_def.get('pop_type', 'unknown'),
            "Employment (1k)": b_def.get('employment_size_val', 0),
            "EPE": pm.get('epe', 0),
            "Profit": pm.get('profit', 0),
            "Margin": pm.get('profit_margin', 0),
            "Input_Cost": pm.get('input_cost', 0),
            "Output_Val": pm.get('output_value', 0),
            "Worker_Food_Cost": pm.get('worker_food_cost', 0),
            "output_price": pm.get('output_price', 0),
        }
        for key, val in pm.items():
            if key in PM_COMPARE_META_KEYS:
                continue
            if isinstance(val, (int, float)) and not pd.isna(val):
                row[f"In_{key}"] = val

        prod = pm.get('produced')
        if prod and not pd.isna(prod):
            row[f"Out_{prod}"] = pm.get('output', 0)
        modifier_food = pm.get('modifier_food_output', 0)
        if modifier_food and modifier_food != 0:
            row["Out_food"] = modifier_food

        rows.append(row)
    return pd.DataFrame(rows)


def analyze_building(building_name):
    comp = building_data.compare_production_methods(
        building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data
    )
    num_slots = max(len(comp['vanilla_slots']), len(comp['modded_slots']))

    slot_dfs = []
    for i in range(num_slots):
        v_df = get_slot_df(building_name, i, is_modded=False, comp=comp)
        m_df = get_slot_df(building_name, i, is_modded=True, comp=comp)
        combined = pd.concat([v_df, m_df], ignore_index=True)
        if combined.empty: continue
        
        # Cleanup zero columns
        for col in combined.columns:
            if col.startswith(('In_', 'Out_')):
                if (combined[col].fillna(0).abs() < 1e-6).all():
                    combined = combined.drop(columns=[col])
        
        meta = ["Version", "PM", "Pop_Type", "Employment (1k)", "EPE"]
        econ = ["Profit", "Margin", "Input_Cost", "Output_Val", "Worker_Food_Cost", "output_price"]
        goods = sorted([c for c in combined.columns if c not in meta + econ])
        
        final_df = combined[meta + econ + goods].fillna(0)
        
        # Explicitly format all numeric columns to 3 decimal places
        float_cols = final_df.select_dtypes(include=[np.number]).columns.tolist()
        format_dict = {col: "{:.3f}" for col in float_cols}
        if 'Margin' in format_dict: 
            format_dict['Margin'] = "{:+.3%}"
        if 'EPE' in format_dict:
            format_dict['EPE'] = "{:+.3%}"
            
        styled = final_df.style.format(format_dict)
        
        # Add gradients for Margin, Profit and EPE
        if 'Margin' in final_df.columns:
            styled = styled.background_gradient(subset=['Margin'], cmap='RdYlGn', vmin=-0.5, vmax=0.5)
        if 'EPE' in final_df.columns:
            # For EPE, positive means we need more efficiency (bad current state), negative means we are already above break-even
            # So we invert the colormap: negative EPE is Green, positive is Red
            styled = styled.background_gradient(subset=['EPE'], cmap='RdYlGn_r', vmin=-0.5, vmax=0.5)
        if 'Profit' in final_df.columns:
            # For profit, we use a divergent map centered around 0
            # We'll use the max absolute profit to center the scale
            max_prof = final_df['Profit'].abs().max()
            styled = styled.background_gradient(subset=['Profit'], cmap='RdYlGn', vmin=-max_prof, vmax=max_prof)
        
        slot_dfs.append(styled)
    return slot_dfs

## Farming Village

In [67]:
farming_village_analysis = analyze_building("farming_village")
for df in farming_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_clay,In_debug_max_profit,In_horses,In_tools,Out_legumes,Out_livestock,Out_maize,Out_millet,Out_olives,Out_potato,Out_rice,Out_wheat
0,Vanilla,farming_village_maintenance,peasants,1.000,-16.667%,0.125,+20.000%,0.625,0.750,0.050,1.500,1.000,0.600,0.000,0.025,0.000,0.500,0.000,0.000,0.000,0.000,0.000,0.000
1,Modded,pp_farming_village_maintenance,peasants,5.000,+43.333%,-0.325,-30.233%,1.075,0.750,0.600,1.500,0.800,0.600,0.000,0.025,0.000,0.500,0.000,0.000,0.000,0.000,0.000,0.000
2,Modded,pp_millet_farm_maintenance,peasants,5.000,+73.077%,-0.475,-42.222%,1.125,0.650,0.600,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.650,0.000,0.000,0.000,0.000
3,Modded,pp_wheat_farm_maintenance,peasants,5.000,+104.545%,-0.575,-51.111%,1.125,0.550,0.600,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.550
4,Modded,pp_maize_farm_maintenance,peasants,5.000,+104.545%,-0.575,-51.111%,1.125,0.550,0.600,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.550,0.000,0.000,0.000,0.000,0.000
5,Modded,pp_rice_farm_maintenance,peasants,5.000,+87.500%,-0.525,-46.667%,1.125,0.600,0.600,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.000,0.000,0.000,0.600,0.000
6,Modded,pp_legumes_farm_maintenance,peasants,5.000,+87.500%,-0.525,-46.667%,1.125,0.600,0.600,1.000,0.000,0.600,0.075,0.100,0.600,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7,Modded,pp_potato_farm_maintenance,peasants,5.000,+87.500%,-0.525,-46.667%,1.125,0.600,0.600,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.000,0.000,0.600,0.000,0.000
8,Modded,pp_olives_farm_maintenance,peasants,5.000,+104.545%,-0.575,-51.111%,1.125,0.550,0.600,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.000,0.550,0.000,0.000,0.000


## Fishing Village

In [68]:
fishing_village_analysis = analyze_building("fishing_village")
for df in fishing_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_debug_max_profit,In_ivory,In_salt,In_tools,Out_fish
0,Vanilla,fishing_village_maintenance,peasants,1.000,-36.000%,0.180,+56.250%,0.320,0.500,0.050,1.000,1.000,0.000,0.060,0.010,0.500
1,Vanilla,arctic_fishing_village_maintenance,peasants,1.000,-38.000%,0.190,+61.290%,0.310,0.500,0.050,1.000,1.000,0.020,0.045,0.000,0.500
2,Modded,pp_fishing_village_maintenance,peasants,5.000,+143.333%,-0.645,-58.904%,1.095,0.450,0.600,1.000,1.000,0.000,0.075,0.040,0.450
3,Modded,pp_arctic_fishing_village_maintenance,peasants,5.000,+133.333%,-0.600,-57.143%,1.050,0.450,0.600,1.000,1.000,0.075,0.030,0.000,0.450


## Forest Village

In [69]:
forest_village_analysis = analyze_building("forest_village")
for df in forest_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_debug_max_profit,In_sand,In_tar,In_tools,In_weaponry,In_wild_game,Out_food,Out_fur,Out_leather,Out_wild_game
0,Vanilla,forest_tannery,peasants,1.000,-16.923%,0.110,+20.370%,0.540,0.650,0.050,3.000,0.000,0.100,0.020,0.000,0.000,0.400,1.000,0.000,0.200,0.000
1,Vanilla,hunting_lodges,peasants,1.000,-20.000%,0.050,+25.000%,0.200,0.250,0.050,1.000,0.400,0.000,0.000,0.000,0.050,0.000,1.000,0.000,0.000,0.200
2,Modded,pp_forest_tannery,peasants,5.000,+81.667%,-0.490,-44.954%,1.090,0.600,0.600,3.000,0.000,0.100,0.020,0.000,0.000,0.400,0.000,0.000,0.200,0.000
3,Modded,pp_hunting_lodges,peasants,5.000,+125.000%,-0.500,-55.556%,0.900,0.400,0.600,1.000,0.400,0.000,0.000,0.000,0.100,0.000,0.000,0.000,0.000,0.400
4,Modded,pp_fur_trapping,peasants,5.000,+98.000%,-0.490,-49.495%,0.990,0.500,0.600,2.000,0.400,0.000,0.000,0.050,0.080,0.000,0.000,0.250,0.000,0.000


## Fruit Orchard

In [70]:
fruit_orchard_analysis = analyze_building("fruit_orchard")
for df in fruit_orchard_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_lumber,Out_fruit
0,Vanilla,fruit_orchard_maintenance,laborers,1.000,+1.667%,-0.005,-1.639%,0.305,0.300,0.050,1.000,0.170,0.300
1,Modded,pp_fruit_orchard_maintenance,peasants,5.000,+77.273%,-0.425,-43.590%,0.975,0.550,0.600,1.000,0.250,0.550


## Sheep Farm

In [71]:
sheep_farms_analysis = analyze_building("sheep_farms")
for df in sheep_farms_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_lumber,In_tools,Out_wool
0,Vanilla,sheep_farm_maintenance,laborers,1.000,-5.600%,0.035,+5.932%,0.590,0.625,0.050,1.250,0.200,0.080,0.500
1,Modded,pp_sheep_farm_maintenance,peasants,5.000,+58.000%,-0.435,-36.709%,1.185,0.750,0.600,1.250,0.250,0.070,0.600


## Cookery

In [72]:
cookery_analysis = analyze_building("cookery")
for df in cookery_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_chili,In_fish,In_fruit,In_legumes,In_livestock,In_maize,In_millet,In_olives,In_pepper,In_potato,In_rice,In_saffron,In_salt,In_wheat,In_wild_game,In_wine,In_wool,Out_victuals
0,Modded,pp_cookery_khichdi,peasants,3.000,+45.051%,-2.230,-31.058%,7.180,4.950,0.120,3.000,0.000,0.000,0.000,2.810,0.000,0.000,0.000,0.000,0.000,0.000,3.300,0.190,0.000,0.000,0.000,0.000,0.000,1.650
1,Modded,pp_cookery_fish_congee,peasants,3.000,+45.051%,-2.230,-31.058%,7.180,4.950,0.120,3.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.190,0.000,3.300,0.000,0.000,0.000,2.810,0.000,0.000,1.650
2,Modded,pp_cookery_mutton_pottage,peasants,3.000,+47.273%,-2.340,-32.099%,7.290,4.950,0.120,3.000,0.000,0.000,0.000,0.000,2.400,0.000,0.000,0.000,0.190,0.000,0.000,0.000,0.000,2.620,0.000,0.000,0.000,1.650
3,Modded,pp_cookery_mediterranean_fish,peasants,3.000,+41.414%,-2.050,-29.286%,7.000,4.950,0.120,3.000,0.000,5.000,0.000,0.000,0.000,0.000,0.000,1.500,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.190,0.000,1.650
4,Modded,pp_cookery_saltfish_porridge,peasants,3.000,+49.697%,-2.460,-33.198%,7.410,4.950,0.120,3.000,0.000,3.740,0.000,0.000,0.000,0.000,2.600,0.000,0.000,0.000,0.000,0.000,0.190,0.000,0.000,0.000,0.000,1.650
5,Modded,pp_cookery_pozole,peasants,3.000,+36.970%,-1.830,-26.991%,6.780,4.950,0.120,3.000,0.400,0.000,0.000,0.000,0.000,2.060,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.600,0.000,0.000,1.650
6,Modded,pp_cookery_pemmican,peasants,3.000,+42.222%,-2.090,-29.688%,7.040,4.950,0.120,3.000,0.000,0.000,0.370,0.000,3.740,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.940,0.000,0.000,1.650
7,Modded,pp_cookery_mutton_and_pease,peasants,3.000,+45.202%,-2.238,-31.130%,7.188,4.950,0.120,3.000,0.000,0.000,0.000,1.680,0.000,0.000,0.000,0.000,0.190,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.550,1.650
8,Modded,pp_cookery_labskaus,peasants,3.000,+42.323%,-2.095,-29.737%,7.045,4.950,0.120,3.000,0.000,1.500,0.000,0.000,2.990,0.000,0.000,0.000,0.000,0.940,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.650
9,Modded,pp_cookery_surstromming,peasants,3.000,+53.939%,-2.670,-35.039%,7.620,4.950,0.120,3.000,0.000,4.680,0.000,0.000,0.000,0.000,0.470,0.000,0.000,0.000,0.000,0.000,0.470,0.000,0.000,0.000,0.000,1.650


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_beer,In_beeswax,In_cloves,In_cocoa,In_coffee,In_fruit,In_liquor,In_livestock,In_sugar,In_tea,In_tobacco,In_wine,Out_victuals
0,Modded,pp_cookery_well_water,peasants,3.000,+0.000%,-0.120,-100.000%,0.120,0.000,0.120,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,Modded,pp_cookery_mead,peasants,3.000,+94.444%,-1.700,-48.571%,3.500,1.800,0.120,3.000,0.940,0.750,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.600
2,Modded,pp_cookery_mulled_wine,peasants,3.000,+63.889%,-1.150,-38.983%,2.950,1.800,0.120,3.000,0.000,0.000,0.190,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.940,0.600
3,Modded,pp_cookery_grog,peasants,3.000,+54.444%,-0.980,-35.252%,2.780,1.800,0.120,3.000,0.000,0.000,0.000,0.000,0.000,0.560,0.840,0.000,0.000,0.000,0.000,0.000,0.600
4,Modded,pp_cookery_chocolate_pot,peasants,3.000,+109.444%,-1.970,-52.255%,3.770,1.800,0.120,3.000,0.000,0.000,0.000,0.560,0.000,0.000,0.000,0.000,0.470,0.000,0.000,0.000,0.600
5,Modded,pp_cookery_coffeehouse_service,peasants,3.000,+78.333%,-1.410,-43.925%,3.210,1.800,0.120,3.000,0.000,0.000,0.000,0.000,0.560,0.000,0.000,0.000,0.000,0.000,0.470,0.000,0.600
6,Modded,pp_cookery_tea_service,peasants,3.000,+47.500%,-0.855,-32.203%,2.655,1.800,0.120,3.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.750,0.000,0.470,0.000,0.000,0.600


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_furniture,In_pottery,In_steel,Out_victuals
0,Modded,pp_cookery_no_packaging,peasants,3.000,+0.000%,-0.120,-100.000%,0.120,0.000,0.120,0.000,0.000,0.000,0.000,0.000
1,Modded,pp_cookery_pottery_jars,peasants,3.000,+108.000%,-1.620,-51.923%,3.120,1.500,0.120,3.000,0.000,3.000,0.000,0.500
2,Modded,pp_cookery_coopered_barrels,peasants,3.000,+108.000%,-1.620,-51.923%,3.120,1.500,0.120,3.000,1.000,0.000,0.000,0.500
3,Modded,pp_cookery_tin_cans,peasants,3.000,+70.667%,-2.120,-41.406%,5.120,3.000,0.120,3.000,0.000,0.000,1.000,1.000


### Cookery — input goods × slot frequency

For each **slot** (column), each cell counts **how many production methods** in that slot use the **good as a non-zero input** (same identification as the `In_*` columns above). Use this to spot goods that appear in many PMs in one slot (potential market pressure) vs goods that are rare.

The **total** column sums across slots: for a given good, that is the number of PM–good pairs (each PM lives in one slot, so it is the total number of cookery PMs that consume that good).

Below the combined matrix, **each slot** also gets its own table: goods sorted by how many PMs in that slot use them (zeros omitted).

In [73]:
def _good_info_for_pm_input_key(key: str, is_modded: bool):
    """Match `BuildingData.compare_production_methods` / input cost: only real goods count."""
    if key in PM_COMPARE_META_KEYS:
        return None
    g = goods_data.get_good(key) if is_modded else goods_data.get_vanilla_good(key)
    if g is None:
        g = goods_data.get_good(key)
    return g


def goods_slot_input_frequency_matrix(building_name: str, is_modded: bool = True) -> pd.DataFrame:
    """
    Rows: goods. Columns: slot_0, slot_1, ... plus 'total'.
    Values: number of PMs in that slot with a non-zero input amount of that good.
    """
    comp = building_data.compare_production_methods(
        building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data
    )
    slots = comp["modded_slots" if is_modded else "vanilla_slots"]
    pairs = []
    for slot_idx, slot in enumerate(slots):
        for _pm_name, pm in slot.items():
            for key, val in pm.items():
                if key in PM_COMPARE_META_KEYS:
                    continue
                if not isinstance(val, (int, float)) or pd.isna(val) or abs(val) < 1e-9:
                    continue
                if _good_info_for_pm_input_key(key, is_modded) is None:
                    continue
                pairs.append((key, slot_idx))
    if not pairs:
        return pd.DataFrame()
    long_df = pd.DataFrame(pairs, columns=["good", "slot"])
    mat = long_df.groupby(["good", "slot"]).size().unstack(fill_value=0).astype(int)
    mat = mat.sort_index()
    mat.columns = [f"slot_{c}" for c in mat.columns]
    mat["total"] = mat.sum(axis=1)
    return mat.sort_values("total", ascending=False)


from IPython.display import Markdown, display


def display_slot_input_frequency_matrix(matrix: pd.DataFrame) -> None:
    """Show full matrix then one sorted table per slot (goods with count > 0 only)."""
    if matrix.empty:
        display(matrix)
        return
    display(matrix.style.format("{:d}").background_gradient(cmap="Blues", axis=None))
    slot_cols = [c for c in matrix.columns if c != "total" and str(c).startswith("slot_")]
    for sc in slot_cols:
        col = matrix[sc]
        slot_df = col[col > 0].sort_values(ascending=False).to_frame(name="PMs_with_input")
        display(Markdown(f"#### {sc}"))
        display(slot_df.style.format({"PMs_with_input": "{:d}"}).background_gradient(cmap="Blues", axis=None))


cookery_input_matrix = goods_slot_input_frequency_matrix("cookery", is_modded=True)
display_slot_input_frequency_matrix(cookery_input_matrix)

,slot_0,slot_1,slot_2,total
good,,,,
fish,4,0,0,4
livestock,3,1,0,4
pepper,3,0,0,3
wild_game,3,0,0,3
legumes,2,0,0,2
millet,2,0,0,2
salt,2,0,0,2
rice,2,0,0,2
wine,1,1,0,2


#### slot_0

,PMs_with_input
good,
fish,4
livestock,3
pepper,3
wild_game,3
legumes,2
millet,2
salt,2
rice,2
wine,1


#### slot_1

,PMs_with_input
good,
livestock,1
wine,1
fruit,1
cloves,1
cocoa,1
beer,1
liquor,1
coffee,1
beeswax,1


#### slot_2

,PMs_with_input
good,
furniture,1
pottery,1
steel,1


In [74]:
# Food contribution by input cost: victuals -> 60 food (tavern)
# Attribute % of finished food to each input good by its share of input cost
from IPython.display import Markdown, display

VICTUALS_TO_FOOD = 60


def _style_input_good_summary(df):
    num_cols = [c for c in df.columns if c != "n_recipes"]
    return (
        df.style.format("{:.2f}", subset=num_cols)
        .format("{:.0f}", subset=["n_recipes"])
        .background_gradient(cmap="RdYlGn", axis=0, subset=num_cols)
        .background_gradient(cmap="YlGnBu", axis=0, subset=["n_recipes"])
    )


def cookery_slot_food_input_full(slot_index: int, title: str):
    """Same analysis as prepared victuals, for any cookery `unique_production_methods` slot index."""
    comp = building_data.compare_production_methods(
        "cookery", goods_data=goods_data, defines_data=defines_data, pop_data=pop_data
    )
    recipe_slot = comp["modded_slots"][slot_index]
    rows_first = []

    for pm_name, pm in recipe_slot.items():
        victuals = pm.get("output", 0.0) if pm.get("produced") == "victuals" else 0.0
        victuals = float(victuals) if victuals else 0.0
        food_out = victuals * VICTUALS_TO_FOOD

        good_costs = []
        for key, val in pm.items():
            if key in PM_COMPARE_META_KEYS:
                continue
            good = key
            amount = float(val) if isinstance(val, (int, float)) and not pd.isna(val) else 0.0
            if amount <= 0:
                continue
            price = 0.0
            if good in goods_data.modded_df.index:
                price = float(goods_data.modded_df.loc[good, "default_market_price"]) if "default_market_price" in goods_data.modded_df.columns else 0.0
            elif good in goods_data.vanilla_df.index:
                price = float(goods_data.vanilla_df.loc[good, "default_market_price"])
            cost = amount * price
            good_costs.append((good, amount, cost))

        total_cost = sum(c for _, _, c in good_costs)
        for good, amount, cost in good_costs:
            pct_of_food = (cost / total_cost * 100) if total_cost > 0 else 0.0
            food_contributed = (pct_of_food / 100) * food_out
            food_per_unit = food_contributed / amount if amount > 0 else np.nan
            food_per_gold = food_contributed / cost if cost > 0 else np.nan
            rows_first.append({
                "PM": pm_name,
                "Input_Good": good,
                "Amount": amount,
                "Cost": cost,
                "Pct_of_food": pct_of_food,
                "Food_contributed": food_contributed,
                "Food_per_unit": food_per_unit,
                "Food_per_gold": food_per_gold,
            })

    df_first = pd.DataFrame(rows_first)
    display(Markdown(title))
    if df_first.empty:
        display(Markdown("_No input rows for this slot (e.g. every PM uses no trade goods)._"))
        return None, None, None

    display(
        df_first.style.format(
            {
                "Amount": "{:.3f}",
                "Cost": "{:.3f}",
                "Pct_of_food": "{:.1f}%",
                "Food_contributed": "{:.2f}",
                "Food_per_unit": "{:.2f}",
                "Food_per_gold": "{:.2f}",
            }
        )
    )
    df_recipe_pivot = df_first.pivot_table(
        index="PM", columns="Input_Good", values="Pct_of_food", aggfunc="first"
    ).fillna(0)
    display(df_recipe_pivot.style.format("{:.1f}%"))

    df_agg = df_first[df_first["Food_per_unit"].notna() & df_first["Food_per_gold"].notna()]
    if df_agg.empty:
        df_second = pd.DataFrame()
    else:
        df_second = (
            df_agg.groupby("Input_Good")
            .agg(
                food_per_unit_avg=("Food_per_unit", "mean"),
                food_per_unit_min=("Food_per_unit", "min"),
                food_per_unit_max=("Food_per_unit", "max"),
                food_per_gold_avg=("Food_per_gold", "mean"),
                food_per_gold_min=("Food_per_gold", "min"),
                food_per_gold_max=("Food_per_gold", "max"),
                n_recipes=("Food_per_unit", "count"),
            )
            .round(2)
        )
    if not df_second.empty:
        display(_style_input_good_summary(df_second))
    return df_first, df_recipe_pivot, df_second


df_first, df_recipe_pivot, df_second = cookery_slot_food_input_full(
    0,
    "### Cookery — prepared victuals (slot 0): food attribution by input cost",
)

### Cookery — prepared victuals (slot 0): food attribution by input cost

,PM,Input_Good,Amount,Cost,Pct_of_food,Food_contributed,Food_per_unit,Food_per_gold
0,pp_cookery_khichdi,rice,3.300,3.300,46.7%,46.27,14.02,14.02
1,pp_cookery_khichdi,legumes,2.810,2.810,39.8%,39.40,14.02,14.02
2,pp_cookery_khichdi,saffron,0.190,0.950,13.5%,13.32,70.11,14.02
3,pp_cookery_fish_congee,rice,3.300,3.300,46.7%,46.27,14.02,14.02
4,pp_cookery_fish_congee,wild_game,2.810,2.810,39.8%,39.40,14.02,14.02
5,pp_cookery_fish_congee,pepper,0.190,0.950,13.5%,13.32,70.11,14.02
6,pp_cookery_mutton_pottage,wheat,2.620,2.620,36.5%,36.18,13.81,13.81
7,pp_cookery_mutton_pottage,livestock,2.400,3.600,50.2%,49.71,20.71,13.81
8,pp_cookery_mutton_pottage,pepper,0.190,0.950,13.2%,13.12,69.04,13.81
9,pp_cookery_mediterranean_fish,fish,5.000,5.000,72.7%,71.95,14.39,14.39


Input_Good,chili,fish,fruit,legumes,livestock,maize,millet,olives,pepper,potato,rice,saffron,salt,wheat,wild_game,wine,wool
PM,,,,,,,,,,,,,,,,,
pp_cookery_fish_congee,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,13.5%,0.0%,46.7%,0.0%,0.0%,0.0%,39.8%,0.0%,0.0%
pp_cookery_khichdi,0.0%,0.0%,0.0%,39.8%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,46.7%,13.5%,0.0%,0.0%,0.0%,0.0%,0.0%
pp_cookery_labskaus,0.0%,21.7%,0.0%,0.0%,64.8%,0.0%,0.0%,0.0%,0.0%,13.6%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
pp_cookery_mediterranean_fish,0.0%,72.7%,0.0%,0.0%,0.0%,0.0%,0.0%,21.8%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,5.5%,0.0%
pp_cookery_mutton_and_pease,0.0%,0.0%,0.0%,23.8%,0.0%,0.0%,0.0%,0.0%,13.4%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,62.8%
pp_cookery_mutton_pottage,0.0%,0.0%,0.0%,0.0%,50.2%,0.0%,0.0%,0.0%,13.2%,0.0%,0.0%,0.0%,0.0%,36.5%,0.0%,0.0%,0.0%
pp_cookery_pemmican,0.0%,0.0%,5.3%,0.0%,81.1%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,13.6%,0.0%,0.0%
pp_cookery_pozole,30.0%,0.0%,0.0%,0.0%,0.0%,30.9%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,39.0%,0.0%,0.0%
pp_cookery_saltfish_porridge,0.0%,51.3%,0.0%,0.0%,0.0%,0.0%,35.7%,0.0%,0.0%,0.0%,0.0%,0.0%,13.0%,0.0%,0.0%,0.0%,0.0%


,food_per_unit_avg,food_per_unit_min,food_per_unit_max,food_per_gold_avg,food_per_gold_min,food_per_gold_max,n_recipes
Input_Good,,,,,,,
chili,74.32,74.32,74.32,14.86,14.86,14.86,1
fish,13.87,13.20,14.39,13.87,13.20,14.39,4
fruit,14.31,14.31,14.31,14.31,14.31,14.31,1
legumes,14.02,14.01,14.02,14.02,14.01,14.02,2
livestock,21.20,20.71,21.46,14.14,13.81,14.31,3
maize,14.86,14.86,14.86,14.86,14.86,14.86,1
millet,13.39,13.20,13.58,13.39,13.20,13.58,2
olives,14.39,14.39,14.39,14.39,14.39,14.39,1
pepper,69.73,69.04,70.11,13.95,13.81,14.02,3


In [75]:
# df_second.sort_values("food_per_unit_avg", ascending=False)
if df_second is not None and not df_second.empty:
    display(
        _style_input_good_summary(df_second.sort_values("food_per_gold_avg", ascending=False))
    )

,food_per_unit_avg,food_per_unit_min,food_per_unit_max,food_per_gold_avg,food_per_gold_min,food_per_gold_max,n_recipes
Input_Good,,,,,,,
chili,74.32,74.32,74.32,14.86,14.86,14.86,1
maize,14.86,14.86,14.86,14.86,14.86,14.86,1
wild_game,14.40,14.02,14.86,14.40,14.02,14.86,3
wine,28.78,28.78,28.78,14.39,14.39,14.39,1
olives,14.39,14.39,14.39,14.39,14.39,14.39,1
fruit,14.31,14.31,14.31,14.31,14.31,14.31,1
potato,14.30,14.30,14.30,14.30,14.30,14.30,1
livestock,21.20,20.71,21.46,14.14,13.81,14.31,3
legumes,14.02,14.01,14.02,14.02,14.01,14.02,2


In [76]:
cookery_slot_food_input_full(
    1,
    "### Cookery — table beverages (slot 1): food attribution by input cost",
)

### Cookery — table beverages (slot 1): food attribution by input cost

,PM,Input_Good,Amount,Cost,Pct_of_food,Food_contributed,Food_per_unit,Food_per_gold
0,pp_cookery_mead,beer,0.940,1.880,55.6%,20.02,21.30,10.65
1,pp_cookery_mead,beeswax,0.750,1.500,44.4%,15.98,21.30,10.65
2,pp_cookery_mulled_wine,wine,0.940,1.880,66.4%,23.92,25.44,12.72
3,pp_cookery_mulled_wine,cloves,0.190,0.950,33.6%,12.08,63.60,12.72
4,pp_cookery_grog,liquor,0.840,2.100,78.9%,28.42,33.83,13.53
5,pp_cookery_grog,fruit,0.560,0.560,21.1%,7.58,13.53,13.53
6,pp_cookery_chocolate_pot,cocoa,0.560,2.240,61.4%,22.09,39.45,9.86
7,pp_cookery_chocolate_pot,sugar,0.470,1.410,38.6%,13.91,29.59,9.86
8,pp_cookery_coffeehouse_service,coffee,0.560,1.680,54.4%,19.57,34.95,11.65
9,pp_cookery_coffeehouse_service,tobacco,0.470,1.410,45.6%,16.43,34.95,11.65


Input_Good,beer,beeswax,cloves,cocoa,coffee,fruit,liquor,livestock,sugar,tea,tobacco,wine
PM,,,,,,,,,,,,
pp_cookery_chocolate_pot,0.0%,0.0%,0.0%,61.4%,0.0%,0.0%,0.0%,0.0%,38.6%,0.0%,0.0%,0.0%
pp_cookery_coffeehouse_service,0.0%,0.0%,0.0%,0.0%,54.4%,0.0%,0.0%,0.0%,0.0%,0.0%,45.6%,0.0%
pp_cookery_grog,0.0%,0.0%,0.0%,0.0%,0.0%,21.1%,78.9%,0.0%,0.0%,0.0%,0.0%,0.0%
pp_cookery_mead,55.6%,44.4%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
pp_cookery_mulled_wine,0.0%,0.0%,33.6%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,66.4%
pp_cookery_tea_service,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,44.4%,0.0%,55.6%,0.0%,0.0%


,food_per_unit_avg,food_per_unit_min,food_per_unit_max,food_per_gold_avg,food_per_gold_min,food_per_gold_max,n_recipes
Input_Good,,,,,,,
beer,21.30,21.30,21.30,10.65,10.65,10.65,1
beeswax,21.30,21.30,21.30,10.65,10.65,10.65,1
cloves,63.60,63.60,63.60,12.72,12.72,12.72,1
cocoa,39.45,39.45,39.45,9.86,9.86,9.86,1
coffee,34.95,34.95,34.95,11.65,11.65,11.65,1
fruit,13.53,13.53,13.53,13.53,13.53,13.53,1
liquor,33.83,33.83,33.83,13.53,13.53,13.53,1
livestock,21.30,21.30,21.30,14.20,14.20,14.20,1
sugar,29.59,29.59,29.59,9.86,9.86,9.86,1


(                                PM Input_Good  Amount  Cost  Pct_of_food  \
 0                  pp_cookery_mead       beer    0.94  1.88        55.62   
 1                  pp_cookery_mead    beeswax    0.75  1.50        44.38   
 2           pp_cookery_mulled_wine       wine    0.94  1.88        66.43   
 3           pp_cookery_mulled_wine     cloves    0.19  0.95        33.57   
 4                  pp_cookery_grog     liquor    0.84  2.10        78.95   
 5                  pp_cookery_grog      fruit    0.56  0.56        21.05   
 6         pp_cookery_chocolate_pot      cocoa    0.56  2.24        61.37   
 7         pp_cookery_chocolate_pot      sugar    0.47  1.41        38.63   
 8   pp_cookery_coffeehouse_service     coffee    0.56  1.68        54.37   
 9   pp_cookery_coffeehouse_service    tobacco    0.47  1.41        45.63   
 10          pp_cookery_tea_service  livestock    0.75  1.12        44.38   
 11          pp_cookery_tea_service        tea    0.47  1.41        55.62   

In [77]:
cookery_slot_food_input_full(
    2,
    "### Cookery — packaging (slot 2): food attribution by input cost",
)

### Cookery — packaging (slot 2): food attribution by input cost

,PM,Input_Good,Amount,Cost,Pct_of_food,Food_contributed,Food_per_unit,Food_per_gold
0,pp_cookery_pottery_jars,pottery,3.000,3.000,100.0%,30.00,10.00,10.00
1,pp_cookery_coopered_barrels,furniture,1.000,3.000,100.0%,30.00,30.00,10.00
2,pp_cookery_tin_cans,steel,1.000,5.000,100.0%,60.00,60.00,12.00


Input_Good,furniture,pottery,steel
PM,,,
pp_cookery_coopered_barrels,100.0%,0.0%,0.0%
pp_cookery_pottery_jars,0.0%,100.0%,0.0%
pp_cookery_tin_cans,0.0%,0.0%,100.0%


,food_per_unit_avg,food_per_unit_min,food_per_unit_max,food_per_gold_avg,food_per_gold_min,food_per_gold_max,n_recipes
Input_Good,,,,,,,
furniture,30.00,30.00,30.00,10.00,10.00,10.00,1
pottery,10.00,10.00,10.00,10.00,10.00,10.00,1
steel,60.00,60.00,60.00,12.00,12.00,12.00,1


(                            PM Input_Good  Amount  Cost  Pct_of_food  \
 0      pp_cookery_pottery_jars    pottery    3.00  3.00       100.00   
 1  pp_cookery_coopered_barrels  furniture    1.00  3.00       100.00   
 2          pp_cookery_tin_cans      steel    1.00  5.00       100.00   
 
    Food_contributed  Food_per_unit  Food_per_gold  
 0             30.00          10.00          10.00  
 1             30.00          30.00          10.00  
 2             60.00          60.00          12.00  ,
 Input_Good                   furniture  pottery  steel
 PM                                                    
 pp_cookery_coopered_barrels     100.00     0.00   0.00
 pp_cookery_pottery_jars           0.00   100.00   0.00
 pp_cookery_tin_cans               0.00     0.00 100.00,
             food_per_unit_avg  food_per_unit_min  food_per_unit_max  \
 Input_Good                                                            
 furniture               30.00              30.00              30.00

## Tavern

In [78]:
tavern_analysis = analyze_building("tavern")
for df in tavern_analysis: display(df)

In [79]:
# Tavern calculator: set parameters and run to get the same-style DF
victuals = 1.0
food_produced = 60.0
employment = 1.0
v_good = goods_data.get_good("victuals") if goods_data else None
victuals_price = float(v_good.get("default_market_price", 3.0)) if v_good is not None else 3.0
production_efficiency = 0.0

# Tavern output is abstract food; value at FOOD_PRICE (defines), not market price
food_good_name = "food"
food_price = float(defines_data.get_define("NMarket", "FOOD_PRICE", 0.05)) if defines_data else 0.05
pop_info = pop_data.get_pop_type("peasants") if pop_data else None
pop_food_consumption = float(pop_info.get("pop_food_consumption", 0.6)) if pop_info is not None else 0.6

worker_food_cost = employment * pop_food_consumption * food_price
victuals_cost = victuals * victuals_price
input_cost = victuals_cost + worker_food_cost
output_value = food_produced * (1 + production_efficiency) * food_price
profit = output_value - input_cost
margin = (output_value / input_cost) - 1 if input_cost > 0 else 0.0
epe = (input_cost / (food_produced * food_price)) - 1 if (food_produced * food_price) > 0 else 0.0

out_col = f"Out_{food_good_name}"
tavern_calc_df = pd.DataFrame([{
    "Version": "Calculator",
    "PM": "tavern_maintenance",
    "Pop_Type": "peasants",
    "Employment (1k)": employment,
    "EPE": epe,
    "Profit": profit,
    "Margin": margin,
    "Input_Cost": input_cost,
    "Output_Val": output_value,
    "Worker_Food_Cost": worker_food_cost,
    "output_price": food_price,
    "In_victuals": victuals,
    out_col: food_produced * (1 + production_efficiency)
}])

meta = ["Version", "PM", "Pop_Type", "Employment (1k)", "EPE"]
econ = ["Profit", "Margin", "Input_Cost", "Output_Val", "Worker_Food_Cost", "output_price"]
goods = ["In_victuals", out_col]
display(tavern_calc_df[meta + econ + goods].fillna(0))


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_victuals,Out_food
0,Calculator,tavern_maintenance,peasants,1.00,-0.57,4.08,1.31,3.12,7.20,0.12,0.12,1.00,60.00


In [80]:
# I think the tavern should be +- 0 at food baseline price? This also means that all global food multipliers will benefit tavern and subsistence
